# `mart_topics`

**Грейн:** одна строка = один образовательный `tag`.

**Источники:**
- `processed/mart_events.parquet` — история действий пользователей и последовательностные признаки;
- `processed/questions_clean.parquet` — вопросы и поле `tags`;
- `processed/lectures_clean.parquet` — справочник лекций и их `tag`.

Шаги:
1. разбирает `questions.tags` в соответствие `question_id → tag`;
2. считает сложность и популярность тем;
3. считает количество лекций и их просмотров;
4. для каждого ответа определяет, была ли **до него** лекция с тем же `tag`;
5. сравнивает accuracy после тематической лекции и без неё;
6. проводит проверки качества данных;
7. сохраняет `question_tags.parquet` и `mart_topics.parquet`.

## 1. Настройки и пути

In [1]:
from pathlib import Path
import os
import shutil
import time

import duckdb
import pandas as pd

# Ноутбук можно запускать из корня проекта или из папки data/.
if (Path("processed") / "mart_events.parquet").exists():
    DATA_DIR = Path(".")
elif (Path("data") / "processed" / "mart_events.parquet").exists():
    DATA_DIR = Path("data")
else:
    raise FileNotFoundError(
        "Не найден processed/mart_events.parquet. "
        "Сначала выполни ноутбук построения mart_events."
    )

PROCESSED_DIR = DATA_DIR / "processed"
CACHE_DIR = DATA_DIR / "mart_cache" / "mart_topics"
TEMP_DIR = CACHE_DIR / "tmp_duckdb"
EVENT_PARTS_DIR = CACHE_DIR / "event_parts"
PARTIAL_DIR = CACHE_DIR / "partial_metrics"

MART_EVENTS_PATH = PROCESSED_DIR / "mart_events.parquet"
QUESTIONS_PATH = PROCESSED_DIR / "questions_clean.parquet"
LECTURES_PATH = PROCESSED_DIR / "lectures_clean.parquet"
QUESTION_TAGS_PATH = PROCESSED_DIR / "question_tags.parquet"
MART_TOPICS_PATH = PROCESSED_DIR / "mart_topics.parquet"

# Все события одного пользователя должны оставаться в одной части.
N_PARTITIONS = 32
RESUME = True

# Для рейтингов/сравнений маленькие выборки лучше отдельно фильтровать.
MIN_ATTEMPTS_FOR_ANALYSIS = 1_000
MIN_ATTEMPTS_PER_LECTURE_GROUP = 500

# На полном Riiid точная медиана может быть заметно тяжелее.
# False = approx_quantile(..., 0.5), True = median(...).
USE_EXACT_MEDIAN = False

for path in [MART_EVENTS_PATH, QUESTIONS_PATH, LECTURES_PATH]:
    assert path.exists(), f"Не найден файл: {path.resolve()}"

CACHE_DIR.mkdir(parents=True, exist_ok=True)
TEMP_DIR.mkdir(parents=True, exist_ok=True)
PARTIAL_DIR.mkdir(parents=True, exist_ok=True)


def sql_path(path: Path) -> str:
    return str(path.resolve()).replace("\\", "/").replace("'", "''")

MART_EVENTS = sql_path(MART_EVENTS_PATH)
QUESTIONS = sql_path(QUESTIONS_PATH)
LECTURES = sql_path(LECTURES_PATH)
QUESTION_TAGS = sql_path(QUESTION_TAGS_PATH)
MART_TOPICS = sql_path(MART_TOPICS_PATH)
TEMP = sql_path(TEMP_DIR)

cpu_count = os.cpu_count() or 4
DUCKDB_THREADS = min(8, max(4, cpu_count))

try:
    import psutil
    total_ram_gb = psutil.virtual_memory().total / 1024**3
    memory_gb = max(2, min(8, int(total_ram_gb * 0.55)))
except Exception:
    total_ram_gb = None
    memory_gb = 4

DUCKDB_MEMORY_LIMIT = f"{memory_gb}GB"

con = duckdb.connect()
con.execute(f"SET threads = {DUCKDB_THREADS}")
con.execute(f"SET memory_limit = '{DUCKDB_MEMORY_LIMIT}'")
con.execute(f"SET temp_directory = '{TEMP}'")
con.execute("SET preserve_insertion_order = false")

print("DuckDB:", duckdb.__version__)
print("Threads:", DUCKDB_THREADS)
print("Memory limit:", DUCKDB_MEMORY_LIMIT)
if total_ram_gb is not None:
    print(f"RAM компьютера: {total_ram_gb:.1f} GB")
print("User partitions:", N_PARTITIONS)
print("Exact median:", USE_EXACT_MEDIAN)


DuckDB: 1.5.5
Threads: 8
Memory limit: 8GB
RAM компьютера: 16.0 GB
User partitions: 32
Exact median: False


## 2. Проверяем схемы входных таблиц

In [2]:
required_mart_cols = {
    "user_id",
    "timestamp",
    "content_type_id",
    "question_id",
    "lecture_id",
    "answered_correctly",
    "prior_question_elapsed_time",
}
required_question_cols = {"question_id", "tags"}
required_lecture_cols = {"lecture_id", "tag"}


def get_columns(parquet_path: str) -> set[str]:
    schema = con.sql(f"DESCRIBE SELECT * FROM read_parquet('{parquet_path}')").df()
    return set(schema["column_name"].tolist())

mart_cols = get_columns(MART_EVENTS)
question_cols = get_columns(QUESTIONS)
lecture_cols = get_columns(LECTURES)

assert required_mart_cols <= mart_cols, f"В mart_events не хватает: {required_mart_cols - mart_cols}"
assert required_question_cols <= question_cols, f"В questions_clean не хватает: {required_question_cols - question_cols}"
assert required_lecture_cols <= lecture_cols, f"В lectures_clean не хватает: {required_lecture_cols - lecture_cols}"

print("✅ Схемы входных таблиц подходят")
mart_rows = con.sql(f"SELECT COUNT(*) FROM read_parquet('{MART_EVENTS}')").fetchone()[0]
print("mart_events rows:", f"{mart_rows:,}")


✅ Схемы входных таблиц подходят
mart_events rows: 101,230,332


## 3. Разбираем `questions.tags`

У вопроса может быть несколько тегов, поэтому создаём промежуточную таблицу:

`question_id | tag`

Один ответ на вопрос с тремя тегами будет учитываться в метриках всех трёх тем. Поэтому сумма `attempts` по `mart_topics` **не обязана** равняться общему числу ответов в `mart_events`.


In [3]:
con.execute(f"""
COPY (
    SELECT DISTINCT
        CAST(q.question_id AS BIGINT) AS question_id,
        TRY_CAST(tag_txt AS INTEGER) AS tag
    FROM read_parquet('{QUESTIONS}') q,
         UNNEST(string_split(trim(CAST(q.tags AS VARCHAR)), ' ')) AS u(tag_txt)
    WHERE q.tags IS NOT NULL
      AND trim(CAST(q.tags AS VARCHAR)) <> ''
      AND TRY_CAST(tag_txt AS INTEGER) IS NOT NULL
)
TO '{QUESTION_TAGS}'
(FORMAT PARQUET, COMPRESSION ZSTD)
""")

question_tags_dq = con.sql(f"""
SELECT
    COUNT(*) AS rows_count,
    COUNT(DISTINCT question_id) AS questions_with_tags,
    COUNT(DISTINCT tag) AS tags_count,
    COUNT(*) FILTER (WHERE tag IS NULL) AS null_tags
FROM read_parquet('{QUESTION_TAGS}')
""").df()

display(question_tags_dq)
assert int(question_tags_dq.loc[0, "null_tags"]) == 0

pair_duplicates = con.sql(f"""
SELECT COUNT(*)
FROM (
    SELECT question_id, tag, COUNT(*) AS cnt
    FROM read_parquet('{QUESTION_TAGS}')
    GROUP BY question_id, tag
    HAVING COUNT(*) > 1
)
""").fetchone()[0]
assert pair_duplicates == 0

print("✅ question_tags построен без дублей")
display(con.sql(f"SELECT * FROM read_parquet('{QUESTION_TAGS}') ORDER BY question_id, tag LIMIT 20").df())


,rows_count,questions_with_tags,tags_count,null_tags
0,30992,13522,188,0


✅ question_tags построен без дублей


,question_id,tag
0,0,38
1,0,51
2,0,131
3,0,162
4,1,36
5,1,81
6,1,131
7,2,92
8,2,101
9,2,131


## 4. Справочники тем

`questions_count` считаем по каталогу вопросов, а `lectures_count` — по каталогу лекций. Это не зависит от того, сколько раз контент встречался в истории пользователей.


In [4]:
con.execute("DROP TABLE IF EXISTS question_tags")
con.execute("DROP TABLE IF EXISTS dim_lectures")
con.execute("DROP TABLE IF EXISTS topic_question_catalog")
con.execute("DROP TABLE IF EXISTS topic_lecture_catalog")
con.execute("DROP TABLE IF EXISTS topic_universe")

con.execute(f"""
CREATE TEMP TABLE question_tags AS
SELECT question_id, tag
FROM read_parquet('{QUESTION_TAGS}')
""")

con.execute(f"""
CREATE TEMP TABLE dim_lectures AS
SELECT
    CAST(lecture_id AS BIGINT) AS lecture_id,
    CAST(tag AS INTEGER) AS tag
FROM read_parquet('{LECTURES}')
""")

con.execute("""
CREATE TEMP TABLE topic_question_catalog AS
SELECT
    tag,
    COUNT(DISTINCT question_id)::BIGINT AS questions_count
FROM question_tags
GROUP BY tag
""")

con.execute("""
CREATE TEMP TABLE topic_lecture_catalog AS
SELECT
    tag,
    COUNT(DISTINCT lecture_id)::BIGINT AS lectures_count
FROM dim_lectures
GROUP BY tag
""")

con.execute("""
CREATE TEMP TABLE topic_universe AS
SELECT tag FROM topic_question_catalog
UNION
SELECT tag FROM topic_lecture_catalog
""")

catalog_dq = con.sql("""
SELECT
    (SELECT COUNT(*) FROM topic_universe) AS topics_total,
    (SELECT COUNT(*) FROM topic_question_catalog) AS question_topics,
    (SELECT COUNT(*) FROM topic_lecture_catalog) AS lecture_topics
""").df()
display(catalog_dq)


,topics_total,question_topics,lecture_topics
0,188,188,151


## 5. Один раз делим `mart_events` по пользователям

Самая тяжёлая часть задачи — определить для каждого ответа, была ли раньше лекция с тем же `tag`.

Чтобы не сортировать сразу весь `mart_events`, делим узкий набор нужных полей по

`bucket = hash(user_id) % 32`.

Все события одного пользователя попадают в один bucket, поэтому историю пользователя можно анализировать независимо внутри каждой части.

Если шаг уже успешно выполнялся, повторный запуск использует cache.


In [5]:
PARTITION_SUCCESS = EVENT_PARTS_DIR / "_SUCCESS"

if not (RESUME and PARTITION_SUCCESS.exists()):
    if EVENT_PARTS_DIR.exists():
        shutil.rmtree(EVENT_PARTS_DIR)
    EVENT_PARTS_DIR.mkdir(parents=True, exist_ok=True)

    parts_out = sql_path(EVENT_PARTS_DIR)
    t0 = time.time()

    con.execute(f"""
    COPY (
        SELECT
            CAST(user_id AS BIGINT) AS user_id,
            CAST(timestamp AS BIGINT) AS timestamp,
            CAST(content_type_id AS TINYINT) AS content_type_id,
            CAST(question_id AS BIGINT) AS question_id,
            CAST(lecture_id AS BIGINT) AS lecture_id,
            CAST(answered_correctly AS TINYINT) AS answered_correctly,
            CAST(prior_question_elapsed_time AS DOUBLE) AS prior_question_elapsed_time,
            CAST(hash(user_id) % {N_PARTITIONS} AS INTEGER) AS bucket
        FROM read_parquet('{MART_EVENTS}')
        WHERE content_type_id IN (0, 1)
    )
    TO '{parts_out}'
    (FORMAT PARQUET, COMPRESSION ZSTD, PARTITION_BY (bucket))
    """)

    PARTITION_SUCCESS.write_text("ok", encoding="utf-8")
    print(f"✅ mart_events partitioned за {(time.time() - t0) / 60:.1f} мин")
else:
    print("✅ Используем готовые event partitions")

bucket_dirs = sorted(EVENT_PARTS_DIR.glob("bucket=*"))
print("Найдено bucket directories:", len(bucket_dirs))
assert len(bucket_dirs) > 0


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

✅ mart_events partitioned за 0.1 мин
Найдено bucket directories: 32


## 6. Для каждого ответа находим предыдущую тематическую лекцию

Для каждой user-partition:

1. question events размножаем по `question_tags`;
2. lecture events сопоставляем с тегом лекции;
3. через `ASOF LEFT JOIN` ищем ближайшую лекцию того же пользователя и того же тега;
4. используем строгое условие `question_timestamp > lecture_timestamp`, поэтому лекция с тем же timestamp не считается просмотренной раньше ответа;
5. сразу агрегируем результат до уровня `tag`, чтобы не сохранять огромную таблицу `answer × tag`.

`students_count` можно потом суммировать между bucket-ами, потому что один `user_id` находится ровно в одном bucket.


In [6]:
def bucket_glob(bucket: int) -> str:
    directory = EVENT_PARTS_DIR / f"bucket={bucket}"
    return sql_path(directory) + "/*.parquet"


def partial_path(bucket: int) -> Path:
    return PARTIAL_DIR / f"topic_partial_{bucket:02d}.parquet"


processed = 0
skipped = 0

for bucket in range(N_PARTITIONS):
    source_dir = EVENT_PARTS_DIR / f"bucket={bucket}"
    if not source_dir.exists():
        continue

    out_path = partial_path(bucket)
    if RESUME and out_path.exists():
        skipped += 1
        continue

    src = bucket_glob(bucket)
    out = sql_path(out_path)
    t0 = time.time()

    con.execute(f"""
    COPY (
        WITH question_attempts AS (
            SELECT
                e.user_id,
                e.timestamp AS question_timestamp,
                qt.tag,
                e.answered_correctly,
                e.prior_question_elapsed_time
            FROM read_parquet('{src}') e
            JOIN question_tags qt
              ON e.question_id = qt.question_id
            WHERE e.content_type_id = 0
              AND e.answered_correctly IN (0, 1)
        ),
        lecture_events AS (
            SELECT
                e.user_id,
                e.timestamp AS lecture_timestamp,
                l.tag
            FROM read_parquet('{src}') e
            JOIN dim_lectures l
              ON e.lecture_id = l.lecture_id
            WHERE e.content_type_id = 1
        ),
        exposed_answers AS (
            SELECT
                q.user_id,
                q.tag,
                q.answered_correctly,
                q.prior_question_elapsed_time,
                q.question_timestamp,
                l.lecture_timestamp
            FROM question_attempts q
            ASOF LEFT JOIN lecture_events l
              ON q.user_id = l.user_id
             AND q.tag = l.tag
             AND q.question_timestamp > l.lecture_timestamp
        ),
        question_metrics AS (
            SELECT
                tag,
                COUNT(*)::BIGINT AS attempts,
                COUNT(DISTINCT user_id)::BIGINT AS students_count,
                COUNT(*) FILTER (WHERE answered_correctly = 1)::BIGINT AS correct_answers,
                COUNT(*) FILTER (WHERE answered_correctly = 0)::BIGINT AS incorrect_answers,
                COUNT(*) FILTER (WHERE lecture_timestamp IS NOT NULL)::BIGINT AS attempts_after_lecture,
                COUNT(*) FILTER (
                    WHERE lecture_timestamp IS NOT NULL AND answered_correctly = 1
                )::BIGINT AS correct_after_lecture,
                COUNT(*) FILTER (WHERE lecture_timestamp IS NULL)::BIGINT AS attempts_without_lecture,
                COUNT(*) FILTER (
                    WHERE lecture_timestamp IS NULL AND answered_correctly = 1
                )::BIGINT AS correct_without_lecture
            FROM exposed_answers
            GROUP BY tag
        ),
        lecture_metrics AS (
            SELECT
                l.tag,
                COUNT(*)::BIGINT AS lecture_views
            FROM read_parquet('{src}') e
            JOIN dim_lectures l
              ON e.lecture_id = l.lecture_id
            WHERE e.content_type_id = 1
            GROUP BY l.tag
        )
        SELECT
            COALESCE(q.tag, l.tag) AS tag,
            COALESCE(q.attempts, 0)::BIGINT AS attempts,
            COALESCE(q.students_count, 0)::BIGINT AS students_count,
            COALESCE(q.correct_answers, 0)::BIGINT AS correct_answers,
            COALESCE(q.incorrect_answers, 0)::BIGINT AS incorrect_answers,
            COALESCE(l.lecture_views, 0)::BIGINT AS lecture_views,
            COALESCE(q.attempts_after_lecture, 0)::BIGINT AS attempts_after_lecture,
            COALESCE(q.correct_after_lecture, 0)::BIGINT AS correct_after_lecture,
            COALESCE(q.attempts_without_lecture, 0)::BIGINT AS attempts_without_lecture,
            COALESCE(q.correct_without_lecture, 0)::BIGINT AS correct_without_lecture
        FROM question_metrics q
        FULL OUTER JOIN lecture_metrics l
          ON q.tag = l.tag
    )
    TO '{out}'
    (FORMAT PARQUET, COMPRESSION ZSTD)
    """)

    processed += 1
    print(f"bucket {bucket:02d}: ✅ {(time.time() - t0):.1f} sec")

print(f"Готово новых частей: {processed}; пропущено готовых: {skipped}")


bucket 00: ✅ 0.4 sec
bucket 01: ✅ 0.4 sec
bucket 02: ✅ 0.4 sec
bucket 03: ✅ 0.4 sec
bucket 04: ✅ 0.4 sec
bucket 05: ✅ 0.4 sec
bucket 06: ✅ 0.4 sec
bucket 07: ✅ 0.4 sec
bucket 08: ✅ 0.4 sec
bucket 09: ✅ 0.4 sec
bucket 10: ✅ 0.4 sec
bucket 11: ✅ 0.4 sec
bucket 12: ✅ 0.4 sec
bucket 13: ✅ 0.4 sec
bucket 14: ✅ 0.4 sec
bucket 15: ✅ 0.4 sec
bucket 16: ✅ 0.4 sec
bucket 17: ✅ 0.4 sec
bucket 18: ✅ 0.4 sec
bucket 19: ✅ 0.4 sec
bucket 20: ✅ 0.4 sec
bucket 21: ✅ 0.4 sec
bucket 22: ✅ 0.4 sec
bucket 23: ✅ 0.4 sec
bucket 24: ✅ 0.4 sec
bucket 25: ✅ 0.4 sec
bucket 26: ✅ 0.4 sec
bucket 27: ✅ 0.4 sec
bucket 28: ✅ 0.4 sec
bucket 29: ✅ 0.4 sec
bucket 30: ✅ 0.4 sec
bucket 31: ✅ 0.4 sec
Готово новых частей: 32; пропущено готовых: 0


## 7. Объединяем частичные метрики

In [7]:
partial_files = sorted(PARTIAL_DIR.glob("topic_partial_*.parquet"))
assert partial_files, "Не найдены partial metrics"

PARTIAL_GLOB = sql_path(PARTIAL_DIR) + "/topic_partial_*.parquet"

con.execute("DROP TABLE IF EXISTS behavior_metrics")
con.execute(f"""
CREATE TEMP TABLE behavior_metrics AS
SELECT
    tag,
    SUM(attempts)::BIGINT AS attempts,
    SUM(students_count)::BIGINT AS students_count,
    SUM(correct_answers)::BIGINT AS correct_answers,
    SUM(incorrect_answers)::BIGINT AS incorrect_answers,
    SUM(lecture_views)::BIGINT AS lecture_views,
    SUM(attempts_after_lecture)::BIGINT AS attempts_after_lecture,
    SUM(correct_after_lecture)::BIGINT AS correct_after_lecture,
    SUM(attempts_without_lecture)::BIGINT AS attempts_without_lecture,
    SUM(correct_without_lecture)::BIGINT AS correct_without_lecture
FROM read_parquet('{PARTIAL_GLOB}')
GROUP BY tag
""")

display(con.sql("SELECT * FROM behavior_metrics ORDER BY attempts DESC LIMIT 10").df())


,tag,attempts,students_count,correct_answers,incorrect_answers,lecture_views,attempts_after_lecture,correct_after_lecture,attempts_without_lecture,correct_without_lecture
0,92,18814335,357430,12932846,5881489,0,0,0,18814335,12932846
1,38,16796328,269042,12014725,4781603,0,0,0,16796328,12014725
2,81,13943138,345993,9831369,4111769,0,0,0,13943138,9831369
3,29,12400546,349392,8699218,3701328,0,0,0,12400546,8699218
4,162,9681952,318847,7056661,2625291,0,0,0,9681952,7056661
5,102,8991751,342347,6283353,2708398,0,0,0,8991751,6283353
6,143,7244646,340370,5165286,2079360,812,90012,68469,7154634,5096817
7,136,5153159,211465,3276801,1876358,41483,1848865,1337317,3304294,1939484
8,131,4476973,263988,3561810,915163,0,0,0,4476973,3561810
9,62,3339768,231182,2421128,918640,58869,1126925,833896,2212843,1587232


## 8. Median elapsed time

В исходном Riiid доступен `prior_question_elapsed_time`. В `mart_events` он сохранён как исходный признак, поэтому `median_elapsed_time` ниже строится на нём.

это поле относится к предыдущему question bundle, поэтому его нельзя трактовать как точное время ответа на текущий вопрос. 


In [8]:
ALL_EVENT_PARTS = sql_path(EVENT_PARTS_DIR) + "/bucket=*/*.parquet"
median_expr = (
    "median(e.prior_question_elapsed_time)"
    if USE_EXACT_MEDIAN
    else "approx_quantile(e.prior_question_elapsed_time, 0.5)"
)

con.execute("DROP TABLE IF EXISTS topic_elapsed")
con.execute(f"""
CREATE TEMP TABLE topic_elapsed AS
SELECT
    qt.tag,
    {median_expr} AS median_elapsed_time
FROM read_parquet('{ALL_EVENT_PARTS}') e
JOIN question_tags qt
  ON e.question_id = qt.question_id
WHERE e.content_type_id = 0
  AND e.answered_correctly IN (0, 1)
  AND e.prior_question_elapsed_time IS NOT NULL
  AND e.prior_question_elapsed_time >= 0
GROUP BY qt.tag
""")

display(con.sql("SELECT * FROM topic_elapsed ORDER BY tag LIMIT 10").df())


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,tag,median_elapsed_time
0,0,47038.822275
1,1,22754.895902
2,2,17000.000000
3,3,24306.544879
4,4,21021.360715
5,5,21136.993778
6,6,17000.000000
7,7,21896.740254
8,8,22186.221194
9,9,22000.000000


## 9. Собираем базовую `mart_topics`

In [9]:
con.execute("DROP TABLE IF EXISTS mart_topics_base")
con.execute(f"""
CREATE TEMP TABLE mart_topics_base AS
SELECT
    u.tag,
    COALESCE(qc.questions_count, 0)::BIGINT AS questions_count,
    COALESCE(b.attempts, 0)::BIGINT AS attempts,
    COALESCE(b.students_count, 0)::BIGINT AS students_count,
    COALESCE(b.correct_answers, 0)::BIGINT AS correct_answers,
    COALESCE(b.incorrect_answers, 0)::BIGINT AS incorrect_answers,

    CASE
        WHEN COALESCE(b.attempts, 0) > 0
        THEN b.correct_answers::DOUBLE / b.attempts
    END AS accuracy,

    CASE
        WHEN COALESCE(b.attempts, 0) > 0
        THEN 1.0 - (b.correct_answers::DOUBLE / b.attempts)
    END AS difficulty,

    e.median_elapsed_time,

    COALESCE(lc.lectures_count, 0)::BIGINT AS lectures_count,
    COALESCE(b.lecture_views, 0)::BIGINT AS lecture_views,

    COALESCE(b.attempts_after_lecture, 0)::BIGINT AS attempts_after_lecture,
    CASE
        WHEN COALESCE(b.attempts_after_lecture, 0) > 0
        THEN b.correct_after_lecture::DOUBLE / b.attempts_after_lecture
    END AS accuracy_after_lecture,

    COALESCE(b.attempts_without_lecture, 0)::BIGINT AS attempts_without_lecture,
    CASE
        WHEN COALESCE(b.attempts_without_lecture, 0) > 0
        THEN b.correct_without_lecture::DOUBLE / b.attempts_without_lecture
    END AS accuracy_without_lecture,

    CASE
        WHEN COALESCE(b.attempts_after_lecture, 0) > 0
         AND COALESCE(b.attempts_without_lecture, 0) > 0
        THEN
            (b.correct_after_lecture::DOUBLE / b.attempts_after_lecture)
            -
            (b.correct_without_lecture::DOUBLE / b.attempts_without_lecture)
    END AS accuracy_difference,

    (COALESCE(b.attempts, 0) >= {MIN_ATTEMPTS_FOR_ANALYSIS}) AS enough_attempts,

    (
        COALESCE(b.attempts_after_lecture, 0) >= {MIN_ATTEMPTS_PER_LECTURE_GROUP}
        AND
        COALESCE(b.attempts_without_lecture, 0) >= {MIN_ATTEMPTS_PER_LECTURE_GROUP}
    ) AS enough_lecture_comparison

FROM topic_universe u
LEFT JOIN topic_question_catalog qc USING (tag)
LEFT JOIN topic_lecture_catalog lc USING (tag)
LEFT JOIN behavior_metrics b USING (tag)
LEFT JOIN topic_elapsed e USING (tag)
""")

display(con.sql("SELECT * FROM mart_topics_base ORDER BY attempts DESC LIMIT 10").df())


,tag,questions_count,attempts,students_count,correct_answers,incorrect_answers,accuracy,difficulty,median_elapsed_time,lectures_count,lecture_views,attempts_after_lecture,accuracy_after_lecture,attempts_without_lecture,accuracy_without_lecture,accuracy_difference,enough_attempts,enough_lecture_comparison
0,92,2269,18814335,357430,12932846,5881489,0.687393,0.312607,18999.902852,0,0,0,NaN,18814335,0.687393,NaN,True,False
1,38,2256,16796328,269042,12014725,4781603,0.715319,0.284681,18919.890338,0,0,0,NaN,16796328,0.715319,NaN,True,False
2,81,1969,13943138,345993,9831369,4111769,0.705104,0.294896,19663.845230,0,0,0,NaN,13943138,0.705104,NaN,True,False
3,29,1707,12400546,349392,8699218,3701328,0.701519,0.298481,18970.411976,0,0,0,NaN,12400546,0.701519,NaN,True,False
4,162,914,9681952,318847,7056661,2625291,0.728847,0.271153,23071.830134,0,0,0,NaN,9681952,0.728847,NaN,True,False
5,102,789,8991751,342347,6283353,2708398,0.698791,0.301209,18033.907945,0,0,0,NaN,8991751,0.698791,NaN,True,False
6,143,712,7244646,340370,5165286,2079360,0.712980,0.287020,17000.000000,1,812,90012,0.760665,7154634,0.712380,0.048285,True,True
7,136,1033,5153159,211465,3276801,1876358,0.635882,0.364118,23400.398067,7,41483,1848865,0.723318,3304294,0.586959,0.136359,True,True
8,131,650,4476973,263988,3561810,915163,0.795584,0.204416,21227.113412,0,0,0,NaN,4476973,0.795584,NaN,True,False
9,62,194,3339768,231182,2421128,918640,0.724939,0.275061,17000.000000,6,58869,1126925,0.739975,2212843,0.717282,0.022693,True,True


## 10. Определяем `content_gap_flag` по распределениям

Флаг должен выделять темы, у которых одновременно:
- много попыток;
- низкая accuracy;
- мало лекционного контента.

Чтобы не задавать пороги «с потолка», используем распределение самих тем:
- **high attempts** = 75-й перцентиль `attempts`;
- **low accuracy** = 25-й перцентиль `accuracy`;
- **low lectures** = 25-й перцентиль `lectures_count`.

Пороговые значения выводятся ниже и фиксируются в логике сборки.


In [10]:
con.execute("DROP TABLE IF EXISTS content_gap_thresholds")
con.execute("""
CREATE TEMP TABLE content_gap_thresholds AS
SELECT
    quantile_cont(attempts, 0.75) FILTER (WHERE attempts > 0) AS high_attempts_threshold,
    quantile_cont(accuracy, 0.25) FILTER (WHERE accuracy IS NOT NULL) AS low_accuracy_threshold,
    quantile_cont(lectures_count, 0.25) FILTER (WHERE questions_count > 0) AS low_lectures_threshold
FROM mart_topics_base
""")

thresholds = con.sql("SELECT * FROM content_gap_thresholds").df()
display(thresholds)

con.execute("DROP TABLE IF EXISTS mart_topics_final")
con.execute("""
CREATE TEMP TABLE mart_topics_final AS
SELECT
    b.*,
    CASE
        WHEN b.attempts >= t.high_attempts_threshold
         AND b.accuracy <= t.low_accuracy_threshold
         AND b.lectures_count <= t.low_lectures_threshold
        THEN TRUE
        ELSE FALSE
    END AS content_gap_flag
FROM mart_topics_base b
CROSS JOIN content_gap_thresholds t
""")

print("Кандидатов content gap:", con.sql(
    "SELECT COUNT(*) FROM mart_topics_final WHERE content_gap_flag"
).fetchone()[0])


,high_attempts_threshold,low_accuracy_threshold,low_lectures_threshold
0,1104036.5,0.620568,1.0


Кандидатов content gap: 3


## 11. Data Quality проверки

In [11]:
dq = con.sql("""
SELECT
    COUNT(*) AS rows_count,
    COUNT(DISTINCT tag) AS unique_tags,

    COUNT(*) FILTER (
        WHERE accuracy IS NOT NULL AND (accuracy < 0 OR accuracy > 1)
    ) AS bad_accuracy,

    COUNT(*) FILTER (
        WHERE difficulty IS NOT NULL AND (difficulty < 0 OR difficulty > 1)
    ) AS bad_difficulty,

    COUNT(*) FILTER (
        WHERE accuracy IS NOT NULL
          AND ABS(difficulty - (1.0 - accuracy)) > 1e-12
    ) AS bad_difficulty_formula,

    COUNT(*) FILTER (
        WHERE correct_answers + incorrect_answers <> attempts
    ) AS bad_answer_balance,

    COUNT(*) FILTER (
        WHERE attempts_after_lecture + attempts_without_lecture <> attempts
    ) AS bad_lecture_split,

    COUNT(*) FILTER (
        WHERE questions_count < 0
           OR attempts < 0
           OR students_count < 0
           OR correct_answers < 0
           OR incorrect_answers < 0
           OR lectures_count < 0
           OR lecture_views < 0
    ) AS negative_counts
FROM mart_topics_final
""").df()

display(dq)
r = dq.iloc[0]

assert int(r.rows_count) == int(r.unique_tags), "В финальной витрине есть дубли tag"
assert int(r.bad_accuracy) == 0
assert int(r.bad_difficulty) == 0
assert int(r.bad_difficulty_formula) == 0
assert int(r.bad_answer_balance) == 0
assert int(r.bad_lecture_split) == 0
assert int(r.negative_counts) == 0

# Дополнительная проверка справочника lecture_id → tag.
lecture_key_dq = con.sql("""
SELECT COUNT(*)
FROM (
    SELECT lecture_id, COUNT(*) AS cnt
    FROM dim_lectures
    GROUP BY lecture_id
    HAVING COUNT(*) > 1
)
""").fetchone()[0]
assert lecture_key_dq == 0, "В lectures_clean lecture_id не уникален"

print("✅ Базовые DQ-проверки пройдены")


,rows_count,unique_tags,bad_accuracy,bad_difficulty,bad_difficulty_formula,bad_answer_balance,bad_lecture_split,negative_counts
0,188,188,0,0,0,0,0,0


✅ Базовые DQ-проверки пройдены


### Temporal leakage check

В основной сборке `ASOF` использует строгое условие

`question_timestamp > lecture_timestamp`.

Ниже повторяем проверку на одном полном user-bucket: если найдена лекция, её timestamp обязан быть строго меньше timestamp ответа.


In [12]:
DQ_BUCKET = next(
    b for b in range(N_PARTITIONS)
    if (EVENT_PARTS_DIR / f"bucket={b}").exists()
)
DQ_SRC = bucket_glob(DQ_BUCKET)

bad_temporal_matches = con.sql(f"""
WITH question_attempts AS (
    SELECT
        e.user_id,
        e.timestamp AS question_timestamp,
        qt.tag
    FROM read_parquet('{DQ_SRC}') e
    JOIN question_tags qt
      ON e.question_id = qt.question_id
    WHERE e.content_type_id = 0
      AND e.answered_correctly IN (0, 1)
),
lecture_events AS (
    SELECT
        e.user_id,
        e.timestamp AS lecture_timestamp,
        l.tag
    FROM read_parquet('{DQ_SRC}') e
    JOIN dim_lectures l
      ON e.lecture_id = l.lecture_id
    WHERE e.content_type_id = 1
),
matched AS (
    SELECT
        q.question_timestamp,
        l.lecture_timestamp
    FROM question_attempts q
    ASOF LEFT JOIN lecture_events l
      ON q.user_id = l.user_id
     AND q.tag = l.tag
     AND q.question_timestamp > l.lecture_timestamp
)
SELECT COUNT(*)
FROM matched
WHERE lecture_timestamp IS NOT NULL
  AND lecture_timestamp >= question_timestamp
""").fetchone()[0]

assert bad_temporal_matches == 0
print(f"✅ Temporal check пройден на bucket={DQ_BUCKET}")


✅ Temporal check пройден на bucket=0


## 12. Сохраняем `mart_topics.parquet`

In [13]:
if MART_TOPICS_PATH.exists():
    MART_TOPICS_PATH.unlink()

con.execute(f"""
COPY (
    SELECT *
    FROM mart_topics_final
    ORDER BY tag
)
TO '{MART_TOPICS}'
(FORMAT PARQUET, COMPRESSION ZSTD)
""")

rows_written = con.sql(f"SELECT COUNT(*) FROM read_parquet('{MART_TOPICS}')").fetchone()[0]
file_size_mb = MART_TOPICS_PATH.stat().st_size / 1024**2

print(f"✅ Сохранено: {MART_TOPICS_PATH.resolve()}")
print(f"Rows: {rows_written:,}")
print(f"Size: {file_size_mb:.2f} MB")


✅ Сохранено: /Users/kitten/Downloads/data/processed/mart_topics.parquet
Rows: 188
Size: 0.02 MB


## 13. Быстрый просмотр результата

In [14]:
print("Самые популярные темы:")
display(con.sql(f"""
SELECT
    tag, questions_count, attempts, students_count,
    accuracy, difficulty, lectures_count, lecture_views
FROM read_parquet('{MART_TOPICS}')
WHERE enough_attempts
ORDER BY attempts DESC
LIMIT 15
""").df())

print("Кандидаты на content gap:")
display(con.sql(f"""
SELECT
    tag, questions_count, attempts, students_count,
    accuracy, lectures_count, lecture_views,
    content_gap_flag
FROM read_parquet('{MART_TOPICS}')
WHERE content_gap_flag
ORDER BY attempts DESC, accuracy ASC
""").df())

print("Наблюдаемая разница accuracy после лекции и без неё:")
display(con.sql(f"""
SELECT
    tag,
    attempts_after_lecture,
    accuracy_after_lecture,
    attempts_without_lecture,
    accuracy_without_lecture,
    accuracy_difference
FROM read_parquet('{MART_TOPICS}')
WHERE enough_lecture_comparison
ORDER BY ABS(accuracy_difference) DESC
LIMIT 20
""").df())


Самые популярные темы:


,tag,questions_count,attempts,students_count,accuracy,difficulty,lectures_count,lecture_views
0,92,2269,18814335,357430,0.687393,0.312607,0,0
1,38,2256,16796328,269042,0.715319,0.284681,0,0
2,81,1969,13943138,345993,0.705104,0.294896,0,0
3,29,1707,12400546,349392,0.701519,0.298481,0,0
4,162,914,9681952,318847,0.728847,0.271153,0,0
5,102,789,8991751,342347,0.698791,0.301209,0,0
6,143,712,7244646,340370,0.712980,0.287020,1,812
7,136,1033,5153159,211465,0.635882,0.364118,7,41483
8,131,650,4476973,263988,0.795584,0.204416,0,0
9,62,194,3339768,231182,0.724939,0.275061,6,58869


Кандидаты на content gap:


,tag,questions_count,attempts,students_count,accuracy,lectures_count,lecture_views,content_gap_flag
0,21,597,1890763,177573,0.615935,0,0,True
1,67,327,1780996,188045,0.543400,1,2062,True
2,103,192,1623958,183856,0.546832,1,838,True


Наблюдаемая разница accuracy после лекции и без неё:


,tag,attempts_after_lecture,accuracy_after_lecture,attempts_without_lecture,accuracy_without_lecture,accuracy_difference
0,23,51142,0.797309,414762,0.394243,0.403066
1,151,51902,0.695908,485278,0.471470,0.224438
2,19,46416,0.616167,263553,0.408901,0.207266
3,103,28884,0.739025,1595074,0.543352,0.195673
4,50,11851,0.698085,792938,0.503510,0.194575
5,123,59929,0.710424,546976,0.521308,0.189116
6,108,138727,0.637172,334073,0.448713,0.188459
7,57,33480,0.697342,218772,0.516446,0.180895
8,94,110685,0.700908,424406,0.523237,0.177671
9,24,7591,0.557371,249022,0.380492,0.176878


## Что получилось

Финальная `mart_topics` содержит по одному `tag` и поля:

- `questions_count`;
- `attempts`;
- `students_count`;
- `correct_answers`;
- `incorrect_answers`;
- `accuracy`;
- `difficulty`;
- `median_elapsed_time`;
- `lectures_count`;
- `lecture_views`;
- `attempts_after_lecture`;
- `accuracy_after_lecture`;
- `attempts_without_lecture`;
- `accuracy_without_lecture`;
- `accuracy_difference`;
- `enough_attempts`;
- `enough_lecture_comparison`;
- `content_gap_flag`.

Для обычной интерпретации `accuracy_difference` не подходит: пользователи, которые смотрят лекции, могут отличаться от остальных по уровню, мотивации, сложности выбранных тем и другим признакам.
